# JEWEL geometry and jet matching

Use a completed JEWEL run to connect the hard scattering's production point and medium path lengths to reconstructed jets. Each event contributes two rows, one for each primary hard leg.

The example uses a matched pair of files from JEWEL 2.6.0 with the custom geometry patch. HepMC contains the event record. The adjacent `.hepmc.geometry.csv` contains the production point, the two shower-start directions, and the straight-ray path estimates. Compressed HepMC works with the same CSV naming convention. This notebook reads existing output and does not run JEWEL.

## Set up the notebook kernel

Activate a hepyy environment with a working cppyy runtime and FastJet available. Run `hepyy install fastjet` if FastJet is missing. See the [hepyy installation guide](https://github.com/matplo/hepyy#installation) for the runtime setup on your platform.

Run these commands from the `hepyy-utils` checkout in that environment.

```sh
python -m pip install -e '.[parquet,jets]' matplotlib jupyterlab
jupyter lab
```

Select the kernel from the same hepyy environment. The `jets` extra installs hepyy. This notebook uses the same loader as `jewel_tables`: it calls `hepyy.load("fastjet")` to load native C++ FastJet through cppyy.

In [ ]:
from hepyy_utils.fastjet import require_fastjet

fastjet = require_fastjet()
print(fastjet.fastjet_version_string())

## Choose the input and subtraction mode

The default prefers the completed local five-event sample when available and otherwise uses the bundled one-event recoil fixture. Set `USE_LOCAL_SAMPLE = False` to use the fixture explicitly, or replace `INPUTS` with your own paths. Keep each HepMC file next to the geometry CSV from the same run.

For utility-layer subtraction, generate with `KEEPRECOILS T`, `WRITESCATCEN T`, `WRITEDUMMIES T`, and `DOSUBTRACTION F`. The subtraction uses the status-3 scattering centers as thermal references and removes the pre-scattering thermal component while retaining the recoil response. Dummy records are excluded from jet clustering. The current reference prescription keeps the tag 2 and tag 3 thermal leftovers. The subtraction convention is recorded in the output metadata.

JEWEL can also subtract at generation time with `DOSUBTRACTION T`. Set `SUBTRACTION = "none"` for those already-subtracted inputs to avoid subtracting the thermal component twice.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = None  # Or Path("/path/to/hepyy-utils") when running from elsewhere.
if REPO_ROOT is None:
    working_directory = Path.cwd().resolve()
    REPO_ROOT = next(
        (
            path
            for path in (working_directory, *working_directory.parents)
            if (path / "examples" / "jewel_geometry_analysis.py").is_file()
        ),
        None,
    )
    if REPO_ROOT is None:
        raise FileNotFoundError("Set REPO_ROOT to your hepyy-utils checkout.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()

sys.path.insert(0, str(REPO_ROOT / "examples"))
from jewel_geometry_analysis import (
    analyze_inputs,
    default_inputs,
    display_columns,
    plot_matches,
)

USE_LOCAL_SAMPLE = True
INPUTS = default_inputs(REPO_ROOT, use_local_sample=USE_LOCAL_SAMPLE)
OUTPUT_DIR = REPO_ROOT / "outputs" / "jewel_geometry_example_notebook"

SUBTRACTION = "utils"
JET_RADIUS = 0.4
MATCH_DELTA_R = 0.3
JET_PT_MIN = 20.0  # GeV
JET_ABS_ETA_MAX = 3.1

for path in INPUTS:
    print(path)

## Reconstruct jets, match the hard legs, then apply analysis cuts

The helper validates each HepMC and geometry pair, builds the jet constituents, and clusters anti-kt jets with hepyy FastJet. The candidate collection includes every jet with positive transverse momentum, independent of pseudorapidity.

The one-to-one assignment maximizes the number of matched legs within the matching radius in pseudorapidity and azimuth, then minimizes the total ΔR. A jet can be assigned at most once. The pT and |η| analysis cuts set the `analysis_selected` flag after matching. Changing those two thresholds leaves the assignments fixed.

In [ ]:
rows = analyze_inputs(
    INPUTS,
    OUTPUT_DIR,
    radius=JET_RADIUS,
    jet_ptmin=JET_PT_MIN,
    jet_eta_max=JET_ABS_ETA_MAX,
    match_delta_r=MATCH_DELTA_R,
    subtraction=SUBTRACTION,
)

## Compare all legs with the analysis selection

The first table retains every leg, including unmatched legs and assignments that fail the analysis cuts. With `matched=True` and `analysis_selected=False`, the assigned jet, ΔR, and path lengths remain available. The second table contains the legs whose assigned jets pass the cuts. The ambiguity flag indicates that a leg had multiple jet candidates or shared a candidate with the other leg.

In [ ]:
from html import escape

from IPython.display import HTML, Image, JSON, display


def show_table(records, caption):
    if not records:
        print(f"{caption} — no rows")
        return
    columns = list(records[0])
    headers = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body_rows = []
    for record in records:
        cells = []
        for column in columns:
            value = record[column]
            if value is None:
                label = "—"
            elif isinstance(value, float):
                label = f"{value:.3f}"
            else:
                label = str(value)
            cells.append(f"<td>{escape(label)}</td>")
        body_rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(
        f"<table><caption>{escape(caption)}</caption>"
        f"<thead><tr>{headers}</tr></thead><tbody>{''.join(body_rows)}</tbody></table>"
    ))


show_table(display_columns(rows), "All hard legs")
selected_rows = [row for row in rows if row["analysis_selected"]]
show_table(display_columns(selected_rows), "Legs with analysis-selected jets")

## Plot the path lengths

`L_hot_fm` is the distance in three spatial dimensions in the collision frame. `L_hot_T_fm` is the transverse distance. Both measure the portion of a straight ray from the production point along the initial hard-leg direction where JEWEL's evolving medium has positive temperature. The match associates that reference ray with a final jet through angular proximity. It does not reconstruct the shower trajectory or establish particle ancestry.

In [ ]:
plot_path = plot_matches(rows, OUTPUT_DIR)
display(Image(filename=str(plot_path)))

## Inspect the saved settings

`analysis_metadata.json` records the FastJet backend, subtraction settings, jet definition, matching radius, analysis cuts, and input provenance. The output directory contains the underlying Parquet tables, both CSV views, and the figure in PNG and PDF formats.

Change `JET_PT_MIN` or `JET_ABS_ETA_MAX` in the configuration and rerun the downstream cells to compare analysis selections for the same matching. Changing `JET_RADIUS` or `MATCH_DELTA_R` can change the assignments.

In [ ]:
import json

metadata_path = OUTPUT_DIR / "analysis_metadata.json"
print(metadata_path)
display(JSON(data=json.loads(metadata_path.read_text(encoding="utf-8"))))